# PageIndex Pipeline

Builds a hierarchical retrieval tree from a markdown file

## Tree structure
- `#` → L1, `##` → L2, `###` → L3 (≥ MIN_L3_OWN_WORDS words), L4 → LLM-discovered (strict, default no split)

## Summary format — single prompt, all levels
Each node summary has two sections:
- **Own content**: exact concepts, features, fields, actions this node's text covers
- **Children**: each child by exact name + what it covers

This format lets the routing model decide in one read whether to stop at this node or go deeper.

## Retrieval — pass-based with STOP/DESCEND/DROP
Each pass is one LLM call. The model sees node summaries and decides per node:
- **STOP** — this node's text is sufficient evidence (fully or partially)
- **DESCEND** — need to look at children
- **DROP** — not relevant

Pass 1: all L1 summaries
Pass 2: STOP nodes (re-shown, priority-marked) + children of DESCEND nodes
...continues until all selected nodes say STOP or leaves reached

Children of DESCEND nodes are fetched in parallel (tree lookup, no LLM) before each pass.
Max 3 evidence nodes total. Only the final answer call uses full node text.


## 1) Setup

In [1]:
import sys
import os
import json
import re
import time
import asyncio
import logging
from pathlib import Path
from typing import Any

import nest_asyncio
nest_asyncio.apply()

import litellm
from dotenv import load_dotenv

load_dotenv()

litellm.drop_params = True

PROJECT_ROOT = Path(os.environ.get("PAGEINDEX_ROOT", str(Path.cwd())))

print("Project root:", PROJECT_ROOT)


def llm_completion(model: str, prompt: str) -> str:
    messages = [{"role": "user", "content": prompt}]
    for attempt in range(10):
        try:
            response = litellm.completion(model=model, messages=messages, temperature=0)
            return response.choices[0].message.content or ""
        except Exception as e:
            logging.error(f"llm_completion error: {e}")
            if attempt < 9:
                time.sleep(1)
    return ""


async def llm_acompletion(model: str, prompt: str) -> str:
    messages = [{"role": "user", "content": prompt}]
    for attempt in range(10):
        try:
            response = await litellm.acompletion(model=model, messages=messages, temperature=0)
            return response.choices[0].message.content or ""
        except Exception as e:
            logging.error(f"llm_acompletion error: {e}")
            if attempt < 9:
                await asyncio.sleep(1)
    return ""


def count_tokens(text: str, model: str = None) -> int:
    if not text:
        return 0
    return litellm.token_counter(model=model, text=text)


def llm_completion_stream(model: str, prompt: str):
    """Streaming version — yields text chunks as they arrive.
    Use in the answer call so the user sees output immediately.
    Falls back to non-streaming on error.
    """
    messages = [{"role": "user", "content": prompt}]
    for attempt in range(3):
        try:
            response = litellm.completion(
                model=model, messages=messages, temperature=0, stream=True
            )
            for chunk in response:
                delta = chunk.choices[0].delta.content
                if delta:
                    yield delta
            return
        except Exception as e:
            logging.error(f"llm_completion_stream error: {e}")
            if attempt < 2:
                time.sleep(1)
    # fallback: non-streaming
    yield llm_completion(model, prompt)

16:31:18 - LiteLLM:WARNING: get_model_cost_map.py:271 - LiteLLM: Failed to fetch remote model cost map from https://raw.githubusercontent.com/BerriAI/litellm/main/model_prices_and_context_window.json: [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1006). Falling back to local backup.


Project root: c:\PageIndex\PageIndex


## 2) Config

In [ ]:
MODEL          = "gpt-4o"      # used for answer generation
ROUTING_MODEL  = "gpt-4o-mini"  # routing passes — fast classification, no quality loss vs gpt-4o
MD_PATH = PROJECT_ROOT / "Functional_Requirements_final_with_image_descriptions.md"

# Tree structure
# L3 nodes with fewer own-content words than this are merged into their parent L2
MIN_L3_OWN_WORDS = 45

# L3 nodes with more own-content words than this become candidates for L4 discovery
MIN_L3_FOR_L4_DISCOVERY = 180

# Max L4 children the LLM may return per L3 node
MAX_L4_CHILDREN = 4

# Summary word limits by level
SUMMARY_WORDS = {
    "l4":  80,   # maximum words — write as much as content supports, up to this
    "l3":  110,
    "l2":  150,
    "l1":  200,
}

# Retrieval
MAX_FINAL_NODES      = 3    # max evidence nodes passed to answer

# Build / reuse
OUTPUT_DIR    = PROJECT_ROOT / "PI_outputs_4o_mini"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TREE_JSON_PATH = OUTPUT_DIR / "tree.json"
REBUILD_TREE  = True  
CLEAR_CACHE   = True

print("Markdown:", MD_PATH)
print("Output dir:", OUTPUT_DIR)

Markdown: c:\PageIndex\PageIndex\Functional_Requirements_final_with_image_descriptions.md
Output dir: c:\PageIndex\PageIndex\PI_C_outputs_3_24_4o_mini_2


## 3) Prompts

**SUMMARY_PROMPT** — single prompt for all levels (L1/L2/L3/L4).
Output: `Own content: ...` + `Children: ...`
The two-section format lets the routing model decide stop-or-descend from the summary alone.

**ROUTING_PROMPT** — used for every routing pass.
Input: candidate node summaries (some marked "previously confirmed").
Output: 0–3 nodes each tagged STOP, DESCEND, or DROP.

**ANSWER_PROMPT** — strict, concise. Evidence-only. No inference.


In [3]:
LEAF_SUMMARY_PROMPT = """
Write a retrieval index entry for this document section.
A search system matches user questions against the phrases you write.

Section title: {title}
Section path:  {path}

Section text:
{content}

Write exactly two lines:
Own content summary: <phrases>
Children summary: none

HOW TO WRITE Own content summary:
Format each phrase as: "exact term from text — what it does or what the user gets"
Separate phrases with commas.

The phrases must be specific enough that a user searching for them would expect to land here.

GOOD examples:
- "HR Baseline template — template containing field names like Employee ID, Function, FTE allocation for uploading HRIS data"
- "GL Baseline template — template containing field names like Functional Allocation for uploading GL cost data"
- "Add field button — add custom fields to the HR or GL template before uploading"
- "duplicate rows for different peer sets — copy a row in the benchmark template to define values for multiple peer groups"
- "primary default peer set — first row for each taxonomy level is treated as the default peer set"
- "KPI values check — validates uploaded KPIs against the standard list and flags unknown KPIs"
- "data type mismatch — rejects rows where % KPIs have non-% values or vice versa"
- "mandatory fields check — rejects rows where required fields are blank"
- "error file — downloadable Excel listing every failed row with its rejection reason"
- "Employee ID — must be unique across all uploaded rows"
- "FTE allocation — must be less than or equal to 100 percent"

BAD examples (too vague — a user cannot find what they need from these):
- "template — prefilled file"
- "benchmark values — defined per set"
- "validation — checks data"

Rules:
- Only index concepts the text explains in enough detail to answer a question about them
- Cover every concept — missing one means queries about it return no result
- Do not infer or add context not in the text
- Up to {max_words} words total
""".strip()


PARENT_SUMMARY_PROMPT = """
Write a retrieval index entry for this document section.
A search system reads this to decide:
  (1) Does this section's own intro text answer the query? → route here
  (2) Which child section has the answer? → route to that child

Section title: {title}
Section path:  {path}

This section's own intro text (before any sub-sections):
{intro}

Child sections (already indexed — use these to describe each child accurately):
{children_summary}

Write exactly two lines:
Own content summary: <phrases from intro only>
Children summary: <ChildName — search phrases | ChildName — search phrases>

HOW TO WRITE Own content summary:
- Use only the intro text above — nothing from child sections
- Format: "exact term — what it does", comma-separated
- If intro has no indexable concepts: write exactly: Own content summary: none

HOW TO WRITE Children summary:
For each child, write what a user would type to search for that child's content.
Read the child's Own content summary AND its Children summary — then write phrases that capture the most specific and searchable terms.

GOOD examples — specific, searchable, uses terms from the child:
"Page actions — select HRIS and GL input templates, HR Baseline template with Employee ID and Function fields, GL Baseline template with Functional Allocation fields, add custom fields to templates, upload HR and GL baseline data files, validate file columns, check Employee ID uniqueness and FTE allocation limits"
"Benchmark Upload — validate KPI values against standard list, check data type mismatches, check mandatory fields, download error file with rejection reasons, define benchmark values for different peer sets by duplicating rows in template"
"Download Template — download preformatted Excel benchmark template, duplicate rows to define benchmark values for multiple peer sets, first row is default peer set per taxonomy level"
"Scenario management — create and manage SSOI-specific scenarios only, scenarios are global across all SSOI module pages, not for benchmarking scenarios"
"Output — benchmarking output page, Summary view for one KPI and peer set, Detailed view for all KPIs with peer set selectable per row, save new scenarios or overwrite existing ones"
"Template column selection page — choose HR Baseline or GL Baseline template, select relevant columns for the assessment, add extra fields using Add field button"

BAD examples — too vague, a user cannot find specific content:
"Page actions — steps for configuring assessment"
"Benchmark Upload — handles benchmark data"
"Scenario management — manages scenarios"
"Output — shows results"

Separator between children: " | "

EXAMPLE OUTPUT for a routing-hub node (intro has no indexable content):
Own content summary: none
Children summary: Benchmark Upload — KPI values check, data type mismatch check, mandatory fields check, error file download, duplicate rows for different peer sets, default peer set is first row | Explore benchmarks — add benchmarks via Excel, delete benchmarks with minus button, duplicate rows auto-removed | Output — Summary view, Detailed view with peer set per row, save and create scenarios

Up to {max_words} words total. Do not pad.
""".strip()





L4_DISCOVERY_PROMPT = """
Decide whether this node should be split into child nodes.
Default is NO SPLIT. Only split when it clearly improves retrieval precision.

Parent title: {title}
Parent path:  {path}
Parent text: {text}

STEP 1 — both must be true or return should_split: false:
  A. The text covers 2+ distinct sub-topics where a focused question about one
     would get a more precise answer from a dedicated child than from the full parent
  B. Each sub-topic occupies a distinct contiguous block of at least 30 words —
     concepts do not bleed across blocks

STEP 2 — only if Step 1 passed, define each child:
  title:           short noun phrase using exact document terminology
  anchor_phrases:  2–4 verbatim phrases from the text that are the clearest signals for this child
  supporting_span: one verbatim contiguous excerpt ≥ 40 words — copy exactly, no edits
                   If no 40-word verbatim excerpt exists for this child, do not create it

Max {max_children} children. If in any doubt → should_split: false.

Return JSON only:
{{"should_split": false, "children": []}}
OR:
{{"should_split": true, "children": [{{"title": "...", "anchor_phrases": ["..."], "supporting_span": "..."}}]}}
""".strip()


ROUTING_PROMPT = """
Find which document sections answer this query.

QUERY: {query}

CANDIDATES:
{nodes_json}

Each candidate has a "summary" field with two lines:
  Own content summary: concepts this section directly explains
  Children summary: what each child section covers

RULES:
1. If Own content summary directly answers the query → STOP (use this section as evidence)
2. If Own content summary is "none" or doesn't answer, but a child in Children summary does → DESCEND to that child
3. If nothing is relevant → omit

EXAMPLE:
Query: "What validations are applied when uploading benchmark data?"
Candidate summary:
  Own content summary: none
  Children summary: Benchmark Upload — KPI values check, data type mismatch check, mandatory fields check, error file download | Explore benchmarks — add, delete, duplicates
Correct decision: DESCEND [id of this candidate]
Reason: Own content is none, Children names "Benchmark Upload — KPI values check, mandatory fields check" which directly answers the query.

Always prefer the most specific match. If a child names the exact topic → DESCEND rather than STOP at the parent.
Select multiple candidates if the query spans different sections.

Return JSON only — no explanation outside the JSON:
{{"nodes": [{{"id": "0001", "decision": "STOP"}}, {{"id": "0003", "decision": "DESCEND"}}]}}

If nothing is relevant: {{"nodes": []}}
""".strip()



ANSWER_PROMPT = """
Answer the user's question using ONLY the evidence provided below.
You are a helpful chatbot. Be direct and concise. Start your answer immediately — no preamble.

STRICT RULES:
  1. Use only facts explicitly stated in the evidence. No inference, no outside knowledge.
  2. If the evidence contains facts that directly address the question — even with different wording — answer it.
     Example: "users can add custom L2 and L3 levels" answers "how do I add taxonomy functions."
     Example: "create new scenarios, save old scenarios, event-based save" answers "can I create multiple scenarios."
     BUT: "reduce shared services costs" does NOT answer "reduce ERP costs" — ERP is a specific system not mentioned.
     The evidence must address the specific thing asked, not just a related general topic.
  3. If the evidence does not directly address what the question specifically asks, say:
     "This information is not covered in the document."  (no sources line)
     Do not construct an answer by connecting related but distinct concepts.
  4. If evidence partially answers: give what is covered, then on a new line:
     "Not covered by the document: <specific missing part>"
  5. Before writing, read the entire evidence from start to finish.
     Then write — covering ALL concepts relevant to the question, not just the first ones you encounter.
     Do not stop early. If the evidence mentions KPI checks, data type validation, mandatory fields,
     duplicate removal, and error file download — every one of these must appear in the answer.
     Missing a concept from the evidence is an error.
  6. Format: prose by default. Bullets only for 4+ parallel items or sequential steps.
  7. End with sources if evidence was used:
     Sources: [Node XXXX] Path, [Node XXXX] Path

USER QUESTION: {query}

EVIDENCE:
{evidence}
""".strip()


## 4) Helpers

In [4]:
def clean_text(text: str) -> str:
    return re.sub(r"\s+", " ", text.replace("\xa0", " ")).strip()


def clean_summary(text: str) -> str:
    text = (text or "").strip()
    if text.startswith("```"):
        text = re.sub(r"^```[a-zA-Z0-9_+-]*\n?", "", text)
        text = re.sub(r"\n?```$", "", text).strip()
    text = re.sub(r"^\s*(summary|node summary)\s*:\s*", "", text, flags=re.I)
    # If model wrote both labels on one line, force Children summary onto its own line
    text = re.sub(r"(?i)\s+(Children summary:)", r"\n\1", text)
    # Only collapse whitespace within each line, not across lines
    lines = text.splitlines()
    lines = [re.sub(r"\s+", " ", line).strip() for line in lines if line.strip()]
    return "\n".join(lines)


def parse_json_safe(raw: str, default):
    try:
        return json.loads(raw)
    except Exception:
        pass
    m = re.search(r"\{.*\}", raw or "", flags=re.DOTALL)
    if m:
        try:
            return json.loads(m.group(0))
        except Exception:
            return default
    return default


def build_maps(tree: list[dict]) -> tuple[dict, dict]:
    """Build node_map (id→node) and parent_map (id→parent_id) from the tree."""
    node_map: dict[str, dict] = {}
    parent_map: dict[str, str] = {}

    def walk(nodes: list[dict], path: list[str], parent_id: str | None) -> None:
        for node in nodes:
            current_path = path + [node["title"]]
            node["path"] = " > ".join(current_path)
            node_map[node["node_id"]] = node
            if parent_id:
                parent_map[node["node_id"]] = parent_id
            walk(node.get("nodes", []) or [], current_path, node["node_id"])

    walk(tree, [], None)
    return node_map, parent_map


def node_stats(tree: list[dict]) -> dict:
    all_nodes = []
    def walk(nodes):
        for n in nodes:
            all_nodes.append(n)
            walk(n.get("nodes", []) or [])
    walk(tree)
    leaves = [n for n in all_nodes if not n.get("nodes")]
    token_counts = [count_tokens(n.get("text", ""), model=MODEL) for n in leaves]
    return {
        "total_nodes": len(all_nodes),
        "leaf_nodes":  len(leaves),
        "avg_leaf_tokens": round(sum(token_counts) / len(token_counts), 1) if token_counts else 0,
        "max_leaf_tokens": max(token_counts) if token_counts else 0,
    }


## 5) Parse markdown → structural tree

Parses `#`, `##`, `###` headings into a nested node tree.

- **L1** (`#`): always created unless zero own words and zero children
- **L2** (`##`): always created unless zero own words and zero children
- **L3** (`###`): created only if own word count ≥ `MIN_L3_OWN_WORDS` (default 45). Thinner nodes fold into their parent L2 — their text remains in the L2 span and their titles are stored in `all_l3_titles` for routing signal.
- **L4**: not created here — discovered by LLM in the next step

`all_l3_titles` stored on each L2 node and `all_l2_titles` on each L1 node ensure that thin subsection names appear in parent summaries for routing.


In [5]:
HEADING_RE = re.compile(r"^(#{1,4})\s+(.+)$")


def parse_headings(md_text: str) -> tuple[list[str], list[dict]]:
    lines = md_text.splitlines()
    headings = []
    for idx, line in enumerate(lines):
        m = HEADING_RE.match(line.strip())
        if m:
            headings.append({
                "level": len(m.group(1)),
                "title": clean_text(m.group(2)),
                "line":  idx,
            })
    return lines, headings


def span_text(lines: list[str], start: int, end: int) -> str:
    return "\n".join(lines[start:end]).strip()


def own_word_count(lines: list[str], heading_line: int, span_end: int) -> int:
    """Words directly under a heading, before any child heading."""
    count = 0
    for j in range(heading_line + 1, span_end):
        if HEADING_RE.match(lines[j].strip()):
            break
        if lines[j].strip():
            count += len(lines[j].split())
    return count


def build_tree_from_markdown(md_text: str) -> list[dict]:
    """Parse # ## ### headings into a nested node tree. Skips empty placeholder nodes and thin L3 nodes."""
    lines, headings = parse_headings(md_text)
    l1_headings = [h for h in headings if h["level"] == 1]
    tree = []
    node_counter = [1]

    def next_id() -> str:
        nid = str(node_counter[0]).zfill(4)
        node_counter[0] += 1
        return nid

    def next_l1_line(after: int) -> int:
        for h in l1_headings:
            if h["line"] > after:
                return h["line"]
        return len(lines)

    for h1 in l1_headings:
        h1_start = h1["line"]
        h1_end   = next_l1_line(h1_start)

        l2s = [h for h in headings if h["level"] == 2
               and h1_start < h["line"] < h1_end]

        if own_word_count(lines, h1_start, h1_end) == 0 and not l2s:
            continue

        node1 = {
            "node_id": next_id(),
            "level":   1,
            "title":   h1["title"],
            "text":    span_text(lines, h1_start, h1_end),
            "summary": "",
            "nodes":   [],
        }
        # Every ## title under this # — gives the L1 summary full sub-area coverage
        if l2s:
            node1["all_l2_titles"] = [h["title"] for h in l2s]

        for i, h2 in enumerate(l2s):
            h2_start = h2["line"]
            h2_end   = l2s[i + 1]["line"] if i + 1 < len(l2s) else h1_end

            l3s = [h for h in headings if h["level"] == 3
                   and h2_start < h["line"] < h2_end]

            if own_word_count(lines, h2_start, h2_end) == 0 and not l3s:
                continue

            node2 = {
                "node_id": next_id(),
                "level":   2,
                "title":   h2["title"],
                "text":    span_text(lines, h2_start, h2_end),
                "summary": "",
                "nodes":   [],
            }
            # Every ### title under this ## — gives the L2 summary full sub-topic coverage
            if l3s:
                node2["all_l3_titles"] = [h["title"] for h in l3s]

            for j, h3 in enumerate(l3s):
                h3_start = h3["line"]
                h3_end   = l3s[j + 1]["line"] if j + 1 < len(l3s) else h2_end

                own_words = own_word_count(lines, h3_start, h3_end)
                if own_words < MIN_L3_OWN_WORDS:
                    continue

                node3 = {
                    "node_id": next_id(),
                    "level":   3,
                    "title":   h3["title"],
                    "text":    span_text(lines, h3_start, h3_end),
                    "summary": "",
                    "nodes":   [],
                }
                node2["nodes"].append(node3)

            node1["nodes"].append(node2)

        tree.append(node1)

    return tree


md_text = MD_PATH.read_text(encoding="utf-8")


## 6) L4 discovery

For L3 nodes with body word count ≥ `MIN_L3_FOR_L4_DISCOVERY`, the LLM decides whether to split.

**Default: no split.** Children are only created when:
- Each proposed child covers one concept a user would query independently
- Each child has ≥ 40 words of unique supporting text
- Concepts are clearly distinct with no overlap

`MAX_L4_CHILDREN = 4` — hard cap per L3 node.
The prompt instructs the model to return an empty list when in doubt.


In [6]:
def discover_l4_children_sync(
    node: dict,
    model: str,
    node_counter: list[int],
) -> None:
    """Attempt L4 child discovery for a single L3 node. Modifies node in-place."""
    text = node.get("text", "")
    body = "\n".join(text.splitlines()[1:])  # skip heading line
    if len(body.split()) < MIN_L3_FOR_L4_DISCOVERY:
        return

    prompt = L4_DISCOVERY_PROMPT.format(
        title=node["title"],
        path=node.get("path", node["title"]),
        text=clean_text(text),
        max_children=MAX_L4_CHILDREN,
    )

    raw = llm_completion(model, prompt)
    result = parse_json_safe(raw, {"should_split": False, "children": []})

    if not result.get("should_split") or not result.get("children"):
        return

    parent_text = text
    for child_spec in result["children"]:
        title = clean_text(child_spec.get("title", ""))
        span         = clean_text(child_spec.get("supporting_span", ""))
        parent_clean = clean_text(parent_text)
        if not title or not span or span[:60] not in parent_clean:
            continue

        nid = str(node_counter[0]).zfill(4)
        node_counter[0] += 1
        node["nodes"].append({
            "node_id": nid,
            "level":   4,
            "title":   title,
            "text":    span,
            "summary": "",
            "nodes":   [],
        })


def run_l4_discovery(tree: list[dict], model: str, node_counter: list[int]) -> None:
    """Sync L4 discovery — sequential, no async needed for a one-time build."""
    for l1 in tree:
        for l2 in l1.get("nodes", []) or []:
            for l3 in l2.get("nodes", []) or []:
                discover_l4_children_sync(l3, model, node_counter)

## 7) Summary generation — bottom-up

Uses a single `SUMMARY_PROMPT` with a one-shot example for all levels.
The `role` parameter differentiates L1 / L2 / L3 / L4 handling.

**Order** (each level fully completes before the next starts):
1. L4 nodes — from full node text
2. L3 nodes — from full node text
3. L2 nodes — from own intro + subsection names + subsection summaries
4. L1 nodes — from own intro + child names + child summaries + all L3 titles

**Output format**: compact structured index — not prose paragraphs.
Scope line + key concepts + subsection entries. Exact document terminology only.

All calls within each level run in parallel via `asyncio.gather` using
`asyncio.to_thread(llm_completion, ...)` for IPykernel compatibility.


In [7]:
def own_intro_text(node: dict) -> str:
    """Return this node's own intro text for the PARENT summary prompt.

    For L2 nodes: includes intro text PLUS text of thin L3 sections that
    did not form nodes (below MIN_L3_OWN_WORDS). These thin sections have
    no node of their own so this is their only home.

    For L1 nodes: intro text only (before first ## heading).
    L2 sections always form nodes regardless of size — nothing folds into L1.
    """
    full = node.get("text", "")
    lines = full.splitlines()
    level = node.get("level", 1)
    child_titles = {c["title"].strip() for c in (node.get("nodes") or [])}

    # L1 nodes: return only lines before first child heading
    if level == 1:
        intro = []
        for line in lines[1:]:
            if HEADING_RE.match(line.strip()):
                break
            intro.append(line)
        return "\n".join(intro).strip()

    # L2 nodes: include intro + thin L3 sections (those not forming nodes)
    result = []
    current_section_title = None
    current_section_lines = []
    in_child_section = False

    for line in lines[1:]:
        m = HEADING_RE.match(line.strip())
        if m:
            if current_section_title is not None and not in_child_section:
                result.extend(current_section_lines)
            current_section_title = m.group(2).strip()
            in_child_section = current_section_title in child_titles
            current_section_lines = [line]
        else:
            if current_section_title is None:
                result.append(line)
            else:
                current_section_lines.append(line)

    if current_section_title is not None and not in_child_section:
        result.extend(current_section_lines)

    return "\n".join(result).strip()


def children_summary_block(node: dict) -> str:
    """Build the {children_summary} field passed to PARENT summary prompts.
    Passes each child's full summary (both lines) so the parent has complete
    context when writing Children descriptions.
    """
    children   = node.get("nodes", []) or []
    level      = node.get("level", 2)
    all_key    = "all_l3_titles" if level == 2 else "all_l2_titles"
    all_titles = node.get(all_key, []) or []
    node_titles = {child["title"] for child in children}
    thin_titles = [t for t in all_titles if t not in node_titles]

    if not children and not thin_titles:
        return "(none)"

    parts = []
    for child in children:
        summary = child.get("summary", "").strip()
        if not summary:
            continue
        parts.append(f"- {child['title']}\n  {summary}")

    for title in thin_titles:
        parts.append(f"- {title} (short subsection — content is in this node's text)")

    return "\n".join(parts)


def _placeholder_summary(node: dict) -> bool:
    """Return True if node should get a fixed placeholder summary instead of an LLM call."""
    _text = clean_text(node.get("text", ""))
    words = len(_text.split())
    if words < 12:
        return True
    # Only check keywords on short nodes — large nodes may contain these words
    # in their actual content (e.g. "placeholder screens" in UI descriptions)
    if words < 50 and any(p in _text.lower() for p in ["to be picked later", "tbd", "coming soon"]):
        return True
    return False


def summarise_l4_async(node: dict, model: str) -> None:
    """L4 leaf node. Content: verbatim supporting span. No children ever."""
    if _placeholder_summary(node):
        node["summary"] = f'Own content summary: {node["title"]} — placeholder, content not yet defined. Children summary: none'
        return
    max_w = SUMMARY_WORDS["l4"]
    prompt = LEAF_SUMMARY_PROMPT.format(
        title=node["title"],
        path=node.get("path", node["title"]),
        content=clean_text(node.get("text", "")),
        max_words=max_w,
    )
    raw = llm_completion(model, prompt)
    node["summary"] = clean_summary(raw)


def summarise_l3_async(node: dict, model: str) -> None:
    """L3 node. Uses LEAF prompt if no L4 children, PARENT prompt if L4 children exist."""
    if _placeholder_summary(node):
        node["summary"] = f'Own content summary: {node["title"]} — placeholder, content not yet defined. Children summary: none'
        return
    max_w = SUMMARY_WORDS["l3"]
    children = node.get("nodes") or []
    if not children:
        # Leaf L3 — all content is own, no children
        prompt = LEAF_SUMMARY_PROMPT.format(
            title=node["title"],
            path=node.get("path", node["title"]),
            content=clean_text(node.get("text", "")),
            max_words=max_w,
        )
    else:
        # Parent L3 — has L4 children; use intro text + L4 summaries
        intro = clean_text(own_intro_text(node)) or "(none)"
        children_summary = children_summary_block(node)
        prompt = PARENT_SUMMARY_PROMPT.format(
            title=node["title"],
            path=node.get("path", node["title"]),
            intro=intro,
            children_summary=children_summary,
            max_words=max_w,
        )
    raw = llm_completion(model, prompt)
    node["summary"] = clean_summary(raw)


def summarise_l2_async(node: dict, model: str) -> None:
    """L2 node. Uses LEAF prompt if no L3 children, PARENT prompt if L3 children exist."""
    if _placeholder_summary(node):
        node["summary"] = f'Own content summary: {node["title"]} — placeholder, content not yet defined. Children summary: none'
        return
    max_w = SUMMARY_WORDS["l2"]
    children = node.get("nodes") or []
    if not children:
        # Leaf L2 — all content is own text (may include thin L3 content folded in)
        prompt = LEAF_SUMMARY_PROMPT.format(
            title=node["title"],
            path=node.get("path", node["title"]),
            content=clean_text(node.get("text", "")),
            max_words=max_w,
        )
    else:
        # Parent L2 — has L3 children; use intro + thin titles + L3 summaries
        intro = clean_text(own_intro_text(node)) or "(none)"
        children_summary = children_summary_block(node)
        prompt = PARENT_SUMMARY_PROMPT.format(
            title=node["title"],
            path=node.get("path", node["title"]),
            intro=intro,
            children_summary=children_summary,
            max_words=max_w,
        )
    raw = llm_completion(model, prompt)
    node["summary"] = clean_summary(raw)


def summarise_l1_async(node: dict, model: str) -> None:
    """L1 node. Uses LEAF prompt if no L2 children, PARENT prompt if L2 children exist."""
    if _placeholder_summary(node):
        node["summary"] = f'Own content summary: {node["title"]} — placeholder, content not yet defined. Children summary: none'
        return
    max_w = SUMMARY_WORDS["l1"]
    children = node.get("nodes") or []
    if not children:
        # Leaf L1 (e.g. Login, Admin Page) — all content is own text
        prompt = LEAF_SUMMARY_PROMPT.format(
            title=node["title"],
            path=node.get("path", node["title"]),
            content=clean_text(node.get("text", "")),
            max_words=max_w,
        )
    else:
        # Parent L1 — has L2 children; use intro + L2 summaries with L3 titles
        intro = clean_text(own_intro_text(node)) or "(none)"
        children_summary = children_summary_block(node)
        prompt = PARENT_SUMMARY_PROMPT.format(
            title=node["title"],
            path=node.get("path", node["title"]),
            intro=intro,
            children_summary=children_summary,
            max_words=max_w,
        )
    raw = llm_completion(model, prompt)
    node["summary"] = clean_summary(raw)


def generate_all_summaries(tree: list[dict], model: str, max_workers: int = 8) -> None:
    """Generate summaries bottom-up: L4 → L3 → L2 → L1.
    Within each level nodes run in parallel via ThreadPoolExecutor.
    Levels are sequential — children must finish before parents start.
    """
    from concurrent.futures import ThreadPoolExecutor, as_completed

    def _all(level):
        r = []
        def _w(ns):
            for n in ns:
                if n['level'] == level: r.append(n)
                _w(n.get('nodes') or [])
        _w(tree)
        return r

    def _run_level(nodes, fn):
        if not nodes:
            return
        with ThreadPoolExecutor(max_workers=min(max_workers, len(nodes))) as ex:
            futures = {ex.submit(fn, n, model): n for n in nodes}
            for fut in as_completed(futures):
                exc = fut.exception()
                if exc:
                    node = futures[fut]
                    logging.error(f"Summary failed [{node['node_id']}] {node['title']}: {exc}")

    for level, fn in [(4, summarise_l4_async), (3, summarise_l3_async),
                      (2, summarise_l2_async), (1, summarise_l1_async)]:
        nodes = _all(level)
        _run_level(nodes, fn)
        print(f"  L{level}: {len(nodes)}")

## 8) Build or load tree

In [8]:
if CLEAR_CACHE and TREE_JSON_PATH.exists():
    TREE_JSON_PATH.unlink()
    print("Cache cleared:", TREE_JSON_PATH)

print("OUTPUT_DIR    :", OUTPUT_DIR)
print("TREE_JSON_PATH:", TREE_JSON_PATH)
print("REBUILD_TREE  :", REBUILD_TREE)
print()

if not REBUILD_TREE and TREE_JSON_PATH.exists():
    saved = json.loads(TREE_JSON_PATH.read_text(encoding="utf-8"))
    tree        = saved["tree"]
    node_map, parent_map = build_maps(tree)  # built once, reused for all queries

elif not REBUILD_TREE and not TREE_JSON_PATH.exists():
    raise FileNotFoundError(
        f"REBUILD_TREE=False but no tree found at {TREE_JSON_PATH}\n"
        f"Either set REBUILD_TREE=True or point OUTPUT_DIR to the folder containing tree.json"
    )

else:
    build_start = time.perf_counter()

    tree = build_tree_from_markdown(md_text)
    node_map, parent_map = build_maps(tree)
    print(f"Structural parse: {time.perf_counter() - build_start:.2f}s  {node_stats(tree)}")


    max_existing_id = max(int(nid) for nid in node_map.keys())
    node_counter = [max_existing_id + 1]

    run_l4_discovery(tree, MODEL, node_counter)
    node_map, parent_map = build_maps(tree)  # rebuilt after L4 discovery
    print(f"L4 discovery done  {node_stats(tree)}")

    print("Summaries (bottom-up)...")
    generate_all_summaries(tree, MODEL)

    TREE_JSON_PATH.write_text(
        json.dumps({"tree": tree}, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    print(f"Build complete in {time.perf_counter() - build_start:.2f}s")
    print("Saved:", TREE_JSON_PATH)


OUTPUT_DIR    : c:\PageIndex\PageIndex\PI_C_outputs_3_24_4o_mini_2
TREE_JSON_PATH: c:\PageIndex\PageIndex\PI_C_outputs_3_24_4o_mini_2\tree.json
REBUILD_TREE  : True

Structural parse: 0.00s  {'total_nodes': 61, 'leaf_nodes': 47, 'avg_leaf_tokens': 175.6, 'max_leaf_tokens': 646}
L4 discovery done  {'total_nodes': 65, 'leaf_nodes': 49, 'avg_leaf_tokens': 156.1, 'max_leaf_tokens': 631}
Summaries (bottom-up)...
  L4: 4
  L3: 27
  L2: 25
  L1: 9
Build complete in 8.12s
Saved: c:\PageIndex\PageIndex\PI_C_outputs_3_24_4o_mini_2\tree.json


## 9) Retrieval helpers

- `build_candidate_cards`: builds the candidate list for LLM routing calls
- `deduplicate_by_ancestry`: drops parent nodes when a descendant is also selected
- `collect_evidence`: assembles full node text for evidence (max `MAX_FINAL_NODES = 3`)


In [9]:

def deduplicate_by_ancestry(node_ids: list[str], node_map: dict, parent_map: dict) -> list[str]:
    """Keep only the most specific nodes — if both a parent and its descendant are
    selected, drop the parent. The descendant is more specific and its text is
    already contained within the parent's span."""
    id_set = set(node_ids)
    def has_descendant_in_set(nid):
        # Check if any selected node is a descendant of nid
        for candidate in id_set:
            if candidate == nid:
                continue
            current = parent_map.get(candidate)
            while current:
                if current == nid:
                    return True
                current = parent_map.get(current)
        return False
    return [nid for nid in node_ids if not has_descendant_in_set(nid)]


def collect_evidence(node_map: dict, node_ids: list[str]) -> tuple[str, list[dict]]:
    selected, seen = [], set()
    for nid in node_ids:
        node = node_map.get(nid)
        if node and nid not in seen:
            selected.append(node)
            seen.add(nid)
    # Path + raw text only — no summary.
    # Summaries serve routing; sending them to the answer model doubles
    # evidence tokens and increases answer-call latency.
    blocks = []
    for n in selected:
        text = (n.get("text") or "").strip()
        path = n.get("path", n["title"])
        # Include section heading if text starts mid-sentence (no heading line)
        first_line = text.splitlines()[0] if text else ""
        has_heading = first_line.startswith("#")
        header = f"[Node {n['node_id']}] {path}"
        if not has_heading:
            # Prepend path as context so answer model knows what this block is
            text = f"# {path}\n{text}"
        blocks.append(f"{header}\n{text}")
    return "\n\n".join(blocks), selected


## 10) Retrieval — pass-based

**One LLM call per pass. Max passes = tree depth of deepest needed descent. Typically 2–3.**

### Each pass
- `routing_call` builds candidate cards and calls ROUTING_PROMPT
- STOP nodes → added to evidence list, re-shown next pass with priority marker
- DESCEND nodes → children fetched in parallel (pure tree lookup), shown next pass
- DROP / not returned → ignored

### Parallel execution
Children of all DESCEND nodes from one pass are fetched simultaneously before the next LLM call.
The LLM call itself is one call — it sees everything together.

### Evidence assembly
STOP nodes accumulated across passes → deduplicated by ancestry → capped at MAX_FINAL_NODES = 3
Full text assembled only for the answer call.


In [10]:
def routing_call(
    query: str,
    candidate_nodes: list[dict],
    confirmed_ids: set[str],
    model: str = ROUTING_MODEL,
) -> tuple[list[dict], float, any]:
    """Single routing call for one pass. Returns list of {id, decision} dicts."""
    cards = []
    for n in candidate_nodes:
        card = {
            "node_id":      n["node_id"],
            "title":        n["title"],
            "path":         n.get("path", n["title"]),  # needed — document has duplicate heading names
            "summary":      clean_summary(n.get("summary", "") or ""),
            "has_children": bool(n.get("nodes")),
        }
        if n["node_id"] in confirmed_ids:
            card["note"] = "previously confirmed"
        cards.append(card)

    prompt = ROUTING_PROMPT.format(
        query=query,
        nodes_json=json.dumps(cards, indent=2, ensure_ascii=False),
    )
    t0  = time.perf_counter()
    raw = llm_completion(model, prompt)
    elapsed = time.perf_counter() - t0

    parsed = parse_json_safe(raw, {"nodes": []})
    valid_ids = {n["node_id"] for n in candidate_nodes}
    stops    = [d for d in parsed.get("nodes", [])
                if d.get("id") in valid_ids and d.get("decision") == "STOP"]
    descends = [d for d in parsed.get("nodes", [])
                if d.get("id") in valid_ids and d.get("decision") == "DESCEND"
                and d.get("id") not in {s["id"] for s in stops}]
    # Cap STOPs at MAX_FINAL_NODES — they consume evidence slots.
    # Never cap DESCENDs — they are routing instructions, not evidence.
    return stops[:MAX_FINAL_NODES] + descends, elapsed, raw



async def run_query_async(tree: list[dict], query: str, model: str = MODEL, node_map: dict | None = None, parent_map: dict | None = None) -> dict:
    """Pass-based retrieval. Each pass: routing LLM decides STOP or DESCEND per node.
    STOP nodes go to evidence. DESCEND nodes expose their children for next pass.
    Children of all DESCEND nodes are fetched in parallel before the next LLM call.
    Max 3 nodes total. Continues until all nodes say STOP or no children remain.
    All decisions based on summaries only. Full text only in the answer call.
    """
    # Use pre-built maps if provided — avoids rebuilding on every query
    if node_map is None or parent_map is None:
        node_map, parent_map = build_maps(tree)  # local build, not shadowing module-level

    # Start with all L1 nodes
    current_candidates = list(tree)
    confirmed_ids: set[str] = set()   # nodes confirmed STOP in prior passes
    stop_ids:      list[str] = []     # ordered final evidence nodes
    all_logs:      list[dict] = []
    total_routing_time = 0.0
    pass_num = 0

    while current_candidates and len(stop_ids) < MAX_FINAL_NODES:
        pass_num += 1

        # ── Routing call for this pass ────────────────────────────────────────
        _loop = asyncio.get_event_loop()
        decisions, elapsed, raw = await _loop.run_in_executor(
            None, routing_call, query, current_candidates, confirmed_ids, ROUTING_MODEL
        )
        total_routing_time += elapsed

        all_logs.append({
            "pass": pass_num,
            "candidates": [n["node_id"] for n in current_candidates],
            "decisions": decisions,
            "latency_seconds": round(elapsed, 3),
        })

        if not decisions:
            break  # LLM found nothing relevant

        # If a node appears as both STOP and DESCEND, STOP wins — we already have its content
        stop_set    = {d["id"] for d in decisions if d["decision"] == "STOP"}
        new_stop    = list(stop_set)
        new_descend = [d["id"] for d in decisions if d["decision"] == "DESCEND"
                       and d["id"] not in stop_set]

        # Add newly confirmed STOP nodes to evidence (preserve order, no duplicates)
        for nid in new_stop:
            if nid not in stop_ids:
                stop_ids.append(nid)
        confirmed_ids.update(new_stop)

        if len(stop_ids) >= MAX_FINAL_NODES or not new_descend:
            break  # enough evidence or nothing left to descend

        # ── Fetch children of DESCEND nodes in parallel (pure tree lookup) ───
        def get_children(nid: str) -> list[dict]:
            node = node_map.get(nid)
            return node.get("nodes") or [] if node else []

        next_candidates_raw = []
        for nid in new_descend:
            next_candidates_raw.extend(get_children(nid))

        if not next_candidates_raw:
            # DESCEND nodes turned out to be leaves with no children
            # Fall back: treat them as STOP so their text becomes evidence
            for nid in new_descend:
                if nid not in stop_ids:
                    stop_ids.append(nid)
            break

        # Re-add still-relevant confirmed STOP nodes so model sees full picture
        next_raw_ids = {n["node_id"] for n in next_candidates_raw}
        confirmed_still = [
            node_map[nid] for nid in confirmed_ids
            if nid in node_map and nid not in next_raw_ids
        ]
        current_candidates = confirmed_still + next_candidates_raw

    # ── Deduplicate and cap evidence ─────────────────────────────────────────
    stop_ids = list(dict.fromkeys(stop_ids))
    stop_ids = deduplicate_by_ancestry(stop_ids, node_map, parent_map)
    evidence_ids = stop_ids[:MAX_FINAL_NODES]

    # Empty list → collect_evidence returns ("", []) → answer prompt fires "not covered"
    evidence, selected_nodes = collect_evidence(node_map, evidence_ids)

    # ── Answer call — uses MODEL (gpt-4o) for quality ────────────────────────
    t0          = time.perf_counter()
    _loop2 = asyncio.get_event_loop()
    answer      = (await _loop2.run_in_executor(
        None, llm_completion, model, ANSWER_PROMPT.format(query=query, evidence=evidence)
    )).strip()
    answer_time = time.perf_counter() - t0

    return {
        "query":              query,
        "l1_ids":             [d["id"] for d in all_logs[0]["decisions"] if d["decision"] in ("STOP","DESCEND")] if all_logs else [],
        "evidence_node_ids":  evidence_ids,
        "selected_nodes": [
            {"node_id": n["node_id"], "title": n["title"], "path": n.get("path", n["title"])}
            for n in selected_nodes
        ],
        "timing": {
            "routing_seconds": round(total_routing_time, 3),
            "answer_seconds":  round(answer_time, 3),
            "total_seconds":   round(total_routing_time + answer_time, 3),
            "passes":          pass_num,
        },
        "traversal_path": all_logs,
        "answer":              answer,
        "evidence":            evidence,
        "evidence_token_count": count_tokens(evidence, model=MODEL),
    }



def run_query(tree: list[dict], query: str, model: str = MODEL) -> dict:
    """Pass pre-built node_map and parent_map to avoid rebuilding on every query."""
    return asyncio.get_event_loop().run_until_complete(
        run_query_async(tree, query, model, node_map=node_map, parent_map=parent_map)
    )

def run_query_streaming(tree: list[dict], query: str, model: str = MODEL):
    """Streams the answer token by token after routing completes.

    Routing passes run fully via run_query before streaming starts — per-pass
    routing events are not yielded because the sync generator + run_until_complete
    architecture cannot interleave routing and yielding.

    Yields:
        {"type": "token", "text": "..."}   — streamed answer tokens
        {"type": "done",  "result": {...}} — final result dict (same as run_query output)
    """
    result = run_query(tree, query, model)

    # Stream result["answer"] token by token — answer already computed by run_query.
    # Do NOT call llm_completion_stream again — would double cost and may differ.
    for token in result["answer"]:
        yield {"type": "token", "text": token}

    yield {"type": "done", "result": result}

## 11) Test query

In [11]:
# query = "If I upload a file and it has both missing columns and wrong values, what exactly will happen and what should I do next?"

# result = run_query(tree, query, model=MODEL)

# print("L1 ids:",            result["l1_ids"])
# print("Evidence nodes:",    result["evidence_node_ids"])
# print("Timing:",            result["timing"])
# print("Evidence tokens:",   result["evidence_token_count"])
# print()
# print("Selected paths:")
# for n in result["selected_nodes"]:
#     print(f"  [{n['node_id']}] {n['path']}")
# print()
# print("Answer:")
# print(result["answer"])

# # Save
# (OUTPUT_DIR / "qa_result.json").write_text(
#     json.dumps(result, indent=2, ensure_ascii=False), encoding="utf-8"
# )

## 12) Batch query

In [12]:
# TEST_QUERIES = [
#     "I uploaded my file but it's showing errors. Where can I see what went wrong?",
#     "Why am I not seeing some functions in the selection list? Did I miss something in setup?",
#     "I changed something in the setup, do I need to upload the file again?",
#     "I uploaded benchmark data but I don't see anything in the output. What should I check?",
#     "Some rows in my upload got rejected. Can I fix only those or do I need to upload everything again?",
#     "I don't see benchmarks for some levels. Is that because of what I selected earlier?",
#     "I added new benchmark rows, but I'm worried it might overwrite the old ones. Does it do that?",
#     "The output page is showing gaps, but I don't understand how it's calculated. Where is that coming from?",
#     "I changed the taxonomy during setup. Will that affect my uploaded data or results?",
#     "I created a scenario and changed some values, but now I'm not sure what's different from the default. How does this work?",
# ]

TEST_QUERIES = [
    "What validations are applied to benchmark upload data?",
    "How does the Explore Benchmarks flow handle adding, deleting, and duplicate benchmarks?",
    "Explain me what the benchmarking output page shows?",
    "Can I download my benchmarking results?",
    "How do I add functions or sub-functions that are not part of the Bain standard taxonomy in the HRIS/GL data template?",
    "What currency should I use to upload the cost data?",
    "To what extent can we customise the input HRIS and GL file?",
    "Can I create multiple scenarios using benchmarks of different peer-sets?",
    "How do we validate the credibility of external benchmarks?",
    "How can we reduce ERP cost?",
]

async def _run_batch(queries, concurrency: int = 4):
    """Run queries in parallel with a concurrency cap to avoid rate limits."""
    semaphore = asyncio.Semaphore(concurrency)
    async def _run_one(q):
        async with semaphore:
            return await run_query_async(tree, q, MODEL, node_map=node_map, parent_map=parent_map)
    return list(await asyncio.gather(*[_run_one(q) for q in queries]))

batch_start   = time.perf_counter()
batch_results = asyncio.get_event_loop().run_until_complete(_run_batch(TEST_QUERIES))
for r in batch_results:
    print(f"L1: {r['l1_ids']}  Evidence: {r['evidence_node_ids']}  Time: {r['timing']['total_seconds']}s")
    print(r["answer"][:300], "..." if len(r["answer"]) > 300 else "")

print(f"\nBatch done in {time.perf_counter() - batch_start:.1f}s")

(OUTPUT_DIR / "batch_results.json").write_text(
    json.dumps(batch_results, indent=2, ensure_ascii=False), encoding="utf-8"
)
print("Saved:", OUTPUT_DIR / "batch_results.json")

L1: ['0025']  Evidence: ['0028']  Time: 0.89s
The validations applied to benchmark upload data include:

- **KPI Values**: Checks for the standard KPIs and identifies any unknown KPIs present in the data.
- **Data Type Mismatch**: Ensures % values are provided for KPIs with percentage metrics and numbers for numerical metrics.
- **Mandatory Fie ...
L1: ['0025']  Evidence: ['0032']  Time: 0.961s
The Explore Benchmarks flow allows users to manage benchmark data by adding, deleting, and handling duplicates as follows:

- **Adding Benchmarks**: Users can add new benchmarks by clicking the 'Add benchmarks' button, which supports Excel uploads. The uploaded file should contain only new data rows ...
L1: ['0025', '0038']  Evidence: ['0033']  Time: 1.445s
The benchmarking output page shows a comparison between client values and benchmarks, focusing on financial processes. It includes two views: Summary and Detailed. The Summary view displays analysis data for one KPI and peer set at a time, st